# CineMark — Movie Recommendation System
### APSP CAT 2 | Group A11 | Python notebook

**Goal:** clean the supplied movie-feedback data, train two recommendation baselines, evaluate them fairly, and demonstrate how a Markov Chain works on **synthetic sequential data only**.

> **Important data limit:** the real ratings file contains feedback labels but no reliable timestamp or viewing order. Therefore, this notebook does **not** train or evaluate a real-data Markov Chain. The Markov experiment near the end generates its own synthetic sequences and reports them separately.

## What this notebook does
1. Loads the movie catalogue and feedback records.
2. Cleans IDs, labels, duplicate user–movie pairs, and unmatched movies.
3. Creates a reproducible per-user holdout test set.
4. Compares a popularity recommender with a TF–IDF content recommender.
5. Calculates Hit Rate@5, Recall@5, MRR@5, and NDCG@5.
6. Runs a clearly labelled synthetic Markov simulation.
7. Saves result tables and charts under `outputs/`.

**Privacy:** `users.json` / user-profile files are intentionally not loaded. They contain personal information and password fields that are not needed for recommendation.

In [ ]:
from pathlib import Path
import json
import ast
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

RANDOM_SEED = 42
TOP_K = 5
ALPHA = 0.1

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RATINGS_PATH = DATA_DIR / "ratings.json"
MOVIES_PATH = DATA_DIR / "movies.csv"

print("Project folder:", ROOT.resolve())
print("Ratings file exists:", RATINGS_PATH.exists())
print("Movies file exists:", MOVIES_PATH.exists())
if not RATINGS_PATH.exists() or not MOVIES_PATH.exists():
    raise FileNotFoundError(
        "Put ratings.json and movies.csv in a folder named data/ next to this notebook. "
        "See README.md for the exact layout."
    )

## 1. Load the files

The ratings JSON is stored as one JSON object per line (JSON Lines / NDJSON), rather than one large JSON array. Each record has a user ID and a dictionary of movie IDs with feedback labels.

The movie catalogue is a CSV. We use only movie-related fields.

In [ ]:
def read_json_lines(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                warnings.warn(f"Skipping invalid JSON at line {line_no}: {exc}")
    return records

rating_records = read_json_lines(RATINGS_PATH)
movies_raw = pd.read_csv(MOVIES_PATH)

print(f"Rating records: {len(rating_records):,}")
print(f"Movie rows: {len(movies_raw):,}")
print("Movie columns:", list(movies_raw.columns))
display(movies_raw.head(3))

## 2. Inspect and clean the feedback

Labels are interpreted as:
- `1`: positive feedback (“liked”) for this experiment.
- `0`: neutral/unspecified feedback; **not** automatically treated as dislike.
- `-1`: negative feedback.

The field named `submit` is a form-control value, not a movie, so it is ignored. User IDs are converted to trimmed internal IDs for matching, but no user profile fields are used.

In [ ]:
required_movie_cols = {"movie_id", "name"}
missing_cols = required_movie_cols - set(movies_raw.columns)
if missing_cols:
    raise ValueError(f"Movie file is missing required columns: {sorted(missing_cols)}")

movies = movies_raw.copy()
movies["movie_id"] = movies["movie_id"].astype(str).str.strip()
movies = movies[movies["movie_id"].str.startswith("tt", na=False)]
movies = movies.drop_duplicates(subset=["movie_id"], keep="first").reset_index(drop=True)

allowed_labels = {-1, 0, 1}
rating_rows = []
bad_label_count = 0
non_movie_key_count = 0

for record in rating_records:
    user_id = str(record.get("_id", "")).strip()
    rated = record.get("rated", {})
    if not user_id or not isinstance(rated, dict):
        continue
    for movie_id, raw_value in rated.items():
        movie_id = str(movie_id).strip()
        if not movie_id.startswith("tt"):
            non_movie_key_count += 1
            continue
        value = raw_value[0] if isinstance(raw_value, list) and raw_value else raw_value
        try:
            label = int(value)
        except (TypeError, ValueError):
            bad_label_count += 1
            continue
        if label not in allowed_labels:
            bad_label_count += 1
            continue
        rating_rows.append((user_id, movie_id, label))

feedback = pd.DataFrame(rating_rows, columns=["user_id", "movie_id", "label"])
before_dedup = len(feedback)
feedback = feedback.drop_duplicates(["user_id", "movie_id"], keep="last").reset_index(drop=True)
duplicate_pairs_removed = before_dedup - len(feedback)

known_movie_ids = set(movies["movie_id"])
before_match = len(feedback)
feedback = feedback[feedback["movie_id"].isin(known_movie_ids)].reset_index(drop=True)
unmatched_feedback_removed = before_match - len(feedback)

print(f"Valid feedback pairs after cleaning: {len(feedback):,}")
print(f"Unique users: {feedback['user_id'].nunique():,}")
print(f"Unique movies receiving feedback: {feedback['movie_id'].nunique():,}")
print(f"Duplicate user–movie pairs removed: {duplicate_pairs_removed:,}")
print(f"Feedback rows without a catalogue match removed: {unmatched_feedback_removed:,}")
print(f"Non-movie/form keys ignored: {non_movie_key_count:,}")
print(f"Invalid labels ignored: {bad_label_count:,}")
display(feedback["label"].value_counts().sort_index().rename_axis("label").to_frame("count"))

## 3. Data-quality summary

These checks help explain the dataset to the instructor. A model should not silently assume that a neutral label means a dislike, or that the order of keys in a file represents viewing order.

In [ ]:
quality_summary = pd.DataFrame({
    "Check": [
        "Movie rows after ID cleaning",
        "Unique movie IDs",
        "Feedback pairs after cleaning",
        "Unique feedback users",
        "Unique feedback movies",
        "Positive labels (+1)",
        "Neutral labels (0)",
        "Negative labels (-1)",
        "Duplicate pairs removed",
        "Unmatched feedback rows removed",
    ],
    "Value": [
        len(movies),
        movies["movie_id"].nunique(),
        len(feedback),
        feedback["user_id"].nunique(),
        feedback["movie_id"].nunique(),
        int((feedback["label"] == 1).sum()),
        int((feedback["label"] == 0).sum()),
        int((feedback["label"] == -1).sum()),
        duplicate_pairs_removed,
        unmatched_feedback_removed,
    ],
})
display(quality_summary)
quality_summary.to_csv(OUTPUT_DIR / "data_quality_summary.csv", index=False)

## 4. Train/test split

Because the data has no reliable event timestamps, we cannot make a chronological “past predicts future” split. Instead, we use a **per-user holdout**: for each user with at least two positive-feedback movies, one positive movie is hidden for testing and the remaining positive movies are used for training.

This tests whether a recommender can recover a hidden liked movie. It is **not** a next-movie prediction test. The split is repeatable because it uses a fixed random seed.

In [ ]:
positive = feedback[feedback["label"] == 1].copy()
rng = np.random.RandomState(RANDOM_SEED)

train_pairs = []
test_pairs = []
for user_id, group in positive.groupby("user_id", sort=True):
    movie_ids = group["movie_id"].tolist()
    if len(movie_ids) < 2:
        train_pairs.extend((user_id, mid) for mid in movie_ids)
        continue
    holdout_index = rng.randint(0, len(movie_ids))
    test_movie = movie_ids[holdout_index]
    test_pairs.append((user_id, test_movie))
    train_pairs.extend((user_id, mid) for i, mid in enumerate(movie_ids) if i != holdout_index)

train = pd.DataFrame(train_pairs, columns=["user_id", "movie_id"])
test = pd.DataFrame(test_pairs, columns=["user_id", "movie_id"])

if test.empty:
    raise ValueError("Not enough users with at least two positive-feedback movies for evaluation.")

train_seen = train.groupby("user_id")["movie_id"].apply(set).to_dict()
test_truth = test.groupby("user_id")["movie_id"].apply(set).to_dict()
all_movie_ids = movies["movie_id"].tolist()

print(f"Training positive pairs: {len(train):,}")
print(f"Test users: {len(test_truth):,}")
print(f"Test positive targets: {len(test):,}")
print("Seed:", RANDOM_SEED)

## 5. Build recommendation models

### Model A — Popularity baseline
Recommend movies with the most positive feedback in the training set. It is simple and gives us a baseline to beat.

### Model B — Content-based TF–IDF
Combine movie descriptions, genre, language, cast, director and writer into text. TF–IDF converts that text into numeric features. For each user, the system creates a profile from the movies they liked in training and ranks other movies by text similarity.

Movie metadata such as the catalogue's general rating is not used as the target label. Only training feedback is used to build user profiles and popularity scores.

In [ ]:
def parse_listish(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return [str(x) for x in value]
    s = str(value).strip()
    if not s:
        return []
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple, set)):
            return [str(x) for x in parsed]
    except (ValueError, SyntaxError):
        pass
    return [s]

text_fields = ["description", "language", "genre", "cast", "director", "writer"]
for col in text_fields:
    if col not in movies.columns:
        movies[col] = ""

def as_text(value):
    return " ".join(parse_listish(value)) if isinstance(value, str) and value.strip().startswith("[") else ("" if pd.isna(value) else str(value))

for col in ["language", "genre", "cast", "director", "writer"]:
    movies[col] = movies[col].apply(lambda x: " ".join(parse_listish(x)))

movies["description"] = movies["description"].fillna("").astype(str)
movies["name"] = movies["name"].fillna("").astype(str)
movies["content_text"] = (
    movies["name"] + " " +
    movies["description"] + " " +
    movies["genre"] + " " +
    movies["language"] + " " +
    movies["cast"] + " " +
    movies["director"] + " " +
    movies["writer"]
).str.lower()

vectorizer = TfidfVectorizer(stop_words="english", max_features=50000, ngram_range=(1, 2), min_df=1)
tfidf_matrix = vectorizer.fit_transform(movies["content_text"])
movie_to_idx = {mid: i for i, mid in enumerate(movies["movie_id"])}
idx_to_movie = movies["movie_id"].tolist()

popularity_counts = train["movie_id"].value_counts().to_dict()

print("TF–IDF matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))
print("Movies with positive training feedback:", len(popularity_counts))

## 6. Ranking and evaluation metrics

For each test user, recommend five movies that were not present in that user's training history. We measure whether the hidden positive movie appears in the top five and how high it appears.

- **Hit Rate@5:** fraction of test users whose hidden movie appears in the top five.
- **Recall@5:** same as Hit Rate@5 here, because each user has exactly one held-out positive movie.
- **MRR@5:** gives more credit when the hidden movie appears closer to rank 1.
- **NDCG@5:** also rewards a higher position in the recommendation list.

In [ ]:
movie_meta = movies.set_index("movie_id", drop=False)

def recommend_popularity(user_id, k=TOP_K):
    seen = train_seen.get(user_id, set())
    ranked = sorted(
        all_movie_ids,
        key=lambda mid: (popularity_counts.get(mid, 0), mid),
        reverse=True
    )
    return [mid for mid in ranked if mid not in seen][:k]

def recommend_content(user_id, k=TOP_K):
    seen = train_seen.get(user_id, set())
    liked = [mid for mid in seen if mid in movie_to_idx]
    if not liked:
        return recommend_popularity(user_id, k=k)
    liked_idx = [movie_to_idx[mid] for mid in liked]
    user_profile = tfidf_matrix[liked_idx].mean(axis=0)
    scores = linear_kernel(np.asarray(user_profile), tfidf_matrix).ravel()
    order = np.argsort(-scores, kind="stable")
    result = []
    for idx in order:
        mid = idx_to_movie[idx]
        if mid not in seen:
            result.append(mid)
        if len(result) == k:
            break
    return result

def evaluate_recommender(recommend_fn, truth, k=TOP_K):
    rows = []
    for user_id, relevant_items in truth.items():
        recs = recommend_fn(user_id, k=k)
        rank = next((i + 1 for i, mid in enumerate(recs) if mid in relevant_items), None)
        hit = int(rank is not None)
        rows.append({
            "user_id": user_id,
            "hit_rate_at_k": hit,
            "recall_at_k": hit / len(relevant_items) if relevant_items else 0.0,
            "mrr_at_k": (1.0 / rank) if rank is not None else 0.0,
            "ndcg_at_k": (1.0 / np.log2(rank + 1)) if rank is not None else 0.0,
        })
    detail = pd.DataFrame(rows)
    metrics = {
        "Hit Rate@5": detail["hit_rate_at_k"].mean(),
        "Recall@5": detail["recall_at_k"].mean(),
        "MRR@5": detail["mrr_at_k"].mean(),
        "NDCG@5": detail["ndcg_at_k"].mean(),
        "Test users": len(detail),
    }
    return metrics, detail

pop_metrics, pop_detail = evaluate_recommender(recommend_popularity, test_truth)
content_metrics, content_detail = evaluate_recommender(recommend_content, test_truth)

results = pd.DataFrame([
    {"Model": "Popularity baseline", **pop_metrics},
    {"Model": "Content-based TF–IDF", **content_metrics},
])
display(results.style.format({
    "Hit Rate@5": "{:.4f}",
    "Recall@5": "{:.4f}",
    "MRR@5": "{:.4f}",
    "NDCG@5": "{:.4f}",
}))
results.to_csv(OUTPUT_DIR / "model_evaluation_results.csv", index=False)
pop_detail.to_csv(OUTPUT_DIR / "popularity_user_metrics.csv", index=False)
content_detail.to_csv(OUTPUT_DIR / "content_user_metrics.csv", index=False)

In [ ]:
metric_cols = ["Hit Rate@5", "MRR@5", "NDCG@5"]
plot_data = results.set_index("Model")[metric_cols]
ax = plot_data.plot(kind="bar", figsize=(9, 5), rot=0)
ax.set_title("CineMark real-data evaluation (per-user holdout)")
ax.set_ylabel("Score (higher is better)")
ax.set_xlabel("")
ax.legend(loc="best")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "model_comparison.png", dpi=180)
plt.show()

## 7. Inspect example recommendations

This cell displays recommendations for one test user using a masked, notebook-local identifier. Do not publish real user IDs in screenshots or reports. It shows movie titles and scores, not profile data.

In [ ]:
def display_recommendations(recommend_fn, user_id, k=TOP_K):
    recs = recommend_fn(user_id, k=k)
    rows = []
    for rank, mid in enumerate(recs, start=1):
        row = movie_meta.loc[mid]
        rows.append({
            "Rank": rank,
            "Movie": row["name"],
            "Movie ID": mid,
            "Genre": row.get("genre", ""),
            "Training positive count": popularity_counts.get(mid, 0),
        })
    return pd.DataFrame(rows)

example_user = next(iter(test_truth))
print("Example user index (masked):", "user_" + str(list(test_truth).index(example_user) + 1))
display(display_recommendations(recommend_popularity, example_user))
display(display_recommendations(recommend_content, example_user))

## 8. Markov Chain demonstration — synthetic data only

A Markov Chain estimates which movie is likely to come next based on the current movie. It needs ordered sequences. The real CineMark feedback file does not contain reliable viewing timestamps or event order, so this section generates synthetic sequences with a known transition pattern.

The simulation is not CineMark data and its results must not be combined with the real-data metrics above.

In [ ]:
def run_synthetic_markov_simulation(n_users, seed=7, n_movies=60, events_per_user=20, k=5, alpha=0.1):
    rng = np.random.RandomState(seed)
    true_p = np.full((n_movies, n_movies), 0.002)
    for i in range(n_movies):
        candidates = rng.choice([j for j in range(n_movies) if j != i], 4, replace=False)
        true_p[i, candidates] += rng.dirichlet([2, 2, 2, 2]) * 0.9
    np.fill_diagonal(true_p, 0)
    true_p /= true_p.sum(axis=1, keepdims=True)
    start = rng.dirichlet(np.ones(n_movies) * 0.5)

    sequences = []
    for _ in range(n_users):
        seq = [rng.choice(n_movies, p=start)]
        for _ in range(events_per_user - 1):
            seq.append(rng.choice(n_movies, p=true_p[seq[-1]]))
        sequences.append(seq)

    cut = int(events_per_user * 0.8)
    counts = np.zeros((n_movies, n_movies))
    popularity = np.zeros(n_movies)
    for seq in sequences:
        tr = seq[:cut]
        for a, b in zip(tr, tr[1:]):
            if a != b:
                counts[a, b] += 1
        for m in tr:
            popularity[m] += 1

    row_sums = counts.sum(axis=1, keepdims=True)
    learned_p = (counts + alpha) / (row_sums + alpha * n_movies)
    mean_l1_error = np.abs(learned_p - true_p).sum(axis=1).mean()

    hits = {"Popularity": 0, "Markov Chain": 0}
    for seq in sequences:
        tr, target = seq[:cut], seq[cut]
        seen = set(tr)
        def is_hit(scores):
            order = np.argsort(-scores, kind="stable")
            recs = [i for i in order if i not in seen][:k]
            return int(target in recs)
        hits["Popularity"] += is_hit(popularity)
        hits["Markov Chain"] += is_hit(learned_p[tr[-1]] + 1e-9 * popularity)

    return {
        "Synthetic users": n_users,
        "Popularity Hit Rate@5": hits["Popularity"] / n_users,
        "Markov Hit Rate@5": hits["Markov Chain"] / n_users,
        "Mean transition-matrix L1 error": float(mean_l1_error),
    }

simulation_rows = [
    run_synthetic_markov_simulation(n, seed=7 + i)
    for i, n in enumerate([50, 100, 200, 400, 800, 1600])
]
simulation_results = pd.DataFrame(simulation_rows)
display(simulation_results.style.format({
    "Popularity Hit Rate@5": "{:.3%}",
    "Markov Hit Rate@5": "{:.3%}",
    "Mean transition-matrix L1 error": "{:.3f}",
}))
simulation_results.to_csv(OUTPUT_DIR / "synthetic_markov_simulation.csv", index=False)

In [ ]:
ax = simulation_results.plot(
    x="Synthetic users",
    y=["Popularity Hit Rate@5", "Markov Hit Rate@5"],
    marker="o",
    figsize=(9, 5)
)
ax.set_title("Synthetic simulation only — not CineMark user data")
ax.set_ylabel("Hit Rate@5")
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "synthetic_markov_simulation.png", dpi=180)
plt.show()

## 9. Final interpretation

Use the actual tables printed by this notebook when writing the report. Do not copy old metrics if the split, seed, cleaning rules, or code differ.

### What can be concluded
- The popularity and TF–IDF models are evaluated on real supplied feedback with a reproducible holdout.
- The synthetic Markov experiment checks whether a transition model can learn patterns deliberately built into generated sequences.
- The synthetic result is not evidence of real-world CineMark Markov performance.

### What cannot be concluded yet
- We cannot say Markov predicts the next movie for real users until we have reliable ordered interactions.
- A single holdout split does not prove one model is universally better.
- A low Hit Rate@5 does not alone mean the project failed; the catalogue is large and each test user has only one hidden target.

### Next research step
Collect privacy-safe event records such as pseudonymous user ID, movie ID, event type, and timestamp. Then split each user's events chronologically and evaluate Markov against the same baseline on future events.

## 10. Exported files

After running the notebook, the `outputs/` folder should contain:
- `data_quality_summary.csv`
- `model_evaluation_results.csv`
- `popularity_user_metrics.csv`
- `content_user_metrics.csv`
- `model_comparison.png`
- `synthetic_markov_simulation.csv`
- `synthetic_markov_simulation.png`

**Reproducibility:** keep the random seed fixed when comparing models. If you change the split strategy or preprocessing, rerun all cells and use the newly generated metrics.